# Lesson 06 Lab — Attention Acceleration Is an IO Problem

**Puzzle:** If exact attention still performs the same mathematical operation, how can changing the schedule reduce memory and latency?

This notebook retains one complete RTX 5090 execution.


## Why this matters

Naive attention forms scores `QKᵀ`, applies softmax, then multiplies by `V`. The score/probability tensor grows with sequence length squared and may be written to and read from external memory. IO-aware attention tiles Q, K, and V through on-chip storage and maintains online softmax statistics, avoiding full materialization while preserving the exact operation up to floating-point order.


## 0. Predict before running

1. Predict the eager score tensor size for the frozen shape.
2. Predict which route uses less peak allocated memory.
3. Choose a numerical tolerance before reading the output error.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The notebook implements an explicit eager baseline and compares it with PyTorch scaled-dot-product attention. It records output error, CUDA-event latency, and peak allocated memory after resetting the allocator statistic for each route. PyTorch may choose among fused and math backends according to inputs and build, so the recorded evidence names the API and environment rather than claiming a specific FlashAttention kernel without backend diagnostics.

- The quadratic score tensor is an execution choice, not the final output shape.
- Tiling trades on-chip state and recomputation for fewer external reads/writes.
- Exact mathematics does not imply bitwise-identical floating-point evaluation order.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["Q/K/V tiles"] --> B["QKᵀ tile"]
  B --> C["online softmax state"]
  C --> D["accumulate V tile"]
  D -->|"next K/V tile"| B
  D --> E["final output"]
```


## 3. Inspect the visual boundary

This lesson is driven by a Mermaid mechanism map and executable measurements.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 6
LESSON_TITLE = 'Attention Acceleration Is an IO Problem'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260819
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | materialized scores, softmax probabilities, and output matmul |
| Candidate | `scaled_dot_product_attention` with backend chosen by PyTorch |
| Held constant | Q/K/V tensors, scale, dtype, shape, warm-up, and repetitions |
| Measurements | score bytes, latency, peak allocated memory, and maximum output error |
| Evidence | `pytorch-gpu` |

**Experiment:** Compare explicit eager attention with PyTorch SDPA at one fixed BF16 shape.


## 6. Inspect the code

The eager function is intentionally readable. Separate measurement functions reset peak memory, execute one route, synchronize, and retain outputs for the error check. This distinguishes algorithmic intermediate size from allocator evidence.

Do not run until the code matches the frozen table.


In [2]:
dtype = torch.bfloat16
B, H, N, D = 1, 16, 1024, 64
q = torch.randn((B, H, N, D), device=DEVICE, dtype=dtype)
k = torch.randn_like(q)
v = torch.randn_like(q)
scale = 1 / math.sqrt(D)

def eager_attention():
    scores = torch.matmul(q, k.transpose(-2, -1)) * scale
    probs = torch.softmax(scores.float(), dim=-1).to(dtype)
    return torch.matmul(probs, v)

def sdpa_attention():
    return F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=False)

eager_samples = cuda_samples(eager_attention, warmup=3, repeats=15)
sdpa_samples = cuda_samples(sdpa_attention, warmup=3, repeats=15)

def peak_delta(fn):
    torch.cuda.synchronize()
    base = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    out = fn()
    torch.cuda.synchronize()
    return out, (torch.cuda.max_memory_allocated() - base) / 2**20

eager_out, eager_peak = peak_delta(eager_attention)
sdpa_out, sdpa_peak = peak_delta(sdpa_attention)
max_error = float((eager_out.float() - sdpa_out.float()).abs().max().item())
score_mib = B * H * N * N * q.element_size() / 2**20
metrics = {
    "shape": [B, H, N, D], "dtype": str(dtype),
    "score_tensor_mib": score_mib,
    "eager_median_ms": statistics.median(eager_samples),
    "sdpa_median_ms": statistics.median(sdpa_samples),
    "eager_peak_mib": eager_peak,
    "sdpa_peak_mib": sdpa_peak,
    "max_abs_error": max_error,
    "eager_samples_ms": eager_samples, "sdpa_samples_ms": sdpa_samples,
}
analysis = (
    f"The explicit score tensor is {score_mib:.1f} MiB. Eager and SDPA medians were "
    f"{metrics['eager_median_ms']:.3f} and {metrics['sdpa_median_ms']:.3f} ms, with "
    f"{max_error:.6f} maximum absolute output difference. Backend identity is not inferred."
)
print(json.dumps(metrics, indent=2))


{
  "shape": [
    1,
    16,
    1024,
    64
  ],
  "dtype": "torch.bfloat16",
  "score_tensor_mib": 32.0,
  "eager_median_ms": 0.2258560061454773,
  "sdpa_median_ms": 0.041439998894929886,
  "eager_peak_mib": 160.0,
  "sdpa_peak_mib": 2.0634765625,
  "max_abs_error": 0.00390625,
  "eager_samples_ms": [
    0.24534399807453156,
    0.23017600178718567,
    0.22646400332450867,
    0.22748799622058868,
    0.2258560061454773,
    0.22726400196552277,
    0.2252800017595291,
    0.2253119945526123,
    0.22623999416828156,
    0.22495999932289124,
    0.22492800652980804,
    0.2264000028371811,
    0.22416000068187714,
    0.22550399601459503,
    0.22313599288463593
  ],
  "sdpa_samples_ms": [
    0.04335999861359596,
    0.04294399917125702,
    0.0414079986512661,
    0.042527999728918076,
    0.041439998894929886,
    0.04054399952292442,
    0.041471999138593674,
    0.04156799986958504,
    0.04169600084424019,
    0.04134399816393852,
    0.04224000126123428,
    0.041312001645

## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Eager score tensor | 32.000 MiB |
| Eager median | 0.226 ms |
| SDPA median | 0.041 ms |
| Eager peak memory | 160.000 MiB |
| SDPA peak memory | 2.063 MiB |
| Max output error | 0.0039 |


## 8. Explain rather than overclaim

The explicit score tensor is 32.0 MiB. Eager and SDPA medians were 0.226 and 0.041 ms, with 0.003906 maximum absolute output difference. Backend identity is not inferred.

**Evidence boundary:** CUDA work executed through PyTorch. It does not identify an internal instruction, cache event, or proprietary hardware block without additional profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 6, "title": 'Attention Acceleration Is an IO Problem', "environment": ENV,
    "evidence_label": 'pytorch-gpu', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Prefer the IO-aware route only when its numerical contract and supported shape are satisfied; fall back explicitly when backend or precision constraints reject it.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 6,
  "title": "Attention Acceleration Is an IO Problem",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260819
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "shape": [
      1,
      16,
      1024,
      64
    ],
    "dtype": "torch.bfloat16",
    "score_tensor_mib": 32.0,
    "eager_median_ms": 0.2258560061454773,
    "sdpa_median_ms": 0.041439998894929886,
    "eager_peak_mib": 160.0,
    "sdpa_peak_mib": 2.0634765625,
    "max_abs_error": 0.00390625,
    "eager_samples_ms": [
      0.24534399807453156,
      0.23017600178718567,
      0.22646400332450867,
      0.22748799622058868,
      0.2258560061454773,
      0.22726400196552277,
      0.2252800017595291,
      0.2253119945526123,
      0.22623999416828156,
      0.22495999932289124,
      0.22492800652980804,
      0.2264000028371811,
      0.22416000068187714,
 

## 10. Make the decision

> Prefer the IO-aware route only when its numerical contract and supported shape are satisfied; fall back explicitly when backend or precision constraints reject it.

**Failure analysis:** Allocator peak is not physical HBM traffic, and a single shape cannot establish scaling. Backend selection may change with PyTorch, driver, mask, dropout, dtype, or head dimension.


## 11. Extend the evidence

Sweep sequence length and causal/mask modes, record selected SDPA backend diagnostics, and profile DRAM bytes with Nsight Compute.

See [`README.md`](README.md) for the full explanation and references.
